In [1]:
import copy
import json
import numpy as np
import os
import pandas as pd
from scipy.stats import zscore
from statsmodels.stats.inter_rater import fleiss_kappa
from scipy.stats import ranksums

SYSTEMS_PATH = 'data/human_evaluation/ratings/'
REFERENCES_PATH = 'data/human_evaluation/data_REF.json'
DOMAIN_PATH = 'data/human_evaluation/ref2domain.json'

# Parsing the data

In [2]:
rdfs = json.load(open(REFERENCES_PATH))
sys_files = [w for w in os.listdir(SYSTEMS_PATH) if not w.startswith('.')]
domains = json.load(open(DOMAIN_PATH))

doc_id = 1
data = []
for sys_file in sys_files:
    results = json.load(open(os.path.join(SYSTEMS_PATH, sys_file)))
    submission_id = sys_file.replace('.json', '').replace('human_eval_id_', '')

    for sample_id in results:
        entry = [w for w in rdfs['entries'] if list(w.keys())[0] == sample_id][0]
        for worker_id in results[sample_id]:
            assign = results[sample_id][worker_id]
            inp = {
                'id': doc_id,
                'sample_id': sample_id,
                'domain': domains['Id' + str(sample_id)],
                'submission_id': submission_id,
                'worker_id': worker_id,
                'category': entry[sample_id]['category'],
                'size': entry[sample_id]['size'],
            }
            inp.update(assign)
            data.append(inp)
            doc_id += 1

# Inter-rater Agreement
## Fleiss' Kappa

Discretize the ratings in 5 categories

In [3]:
n_cat, max_range = 5, 100 # number of categories
data_discretized = []

ids = [w['id'] for w in data]
correctness = [int((n_cat* w['Correctness']) / (max_range+1)) for w in data]
coverage = [int((n_cat* w['DataCoverage']) / (max_range+1)) for w in data]
fluency = [int((n_cat* w['Fluency']) / (max_range+1)) for w in data]
relevance = [int((n_cat* w['Relevance']) / (max_range+1)) for w in data]
structure = [int((n_cat* w['TextStructure']) / (max_range+1)) for w in data]
    
for i, id_ in enumerate(ids):
    for j, row in enumerate(data):
        if row['id'] == id_:
            row_ = copy.copy(row)
            row_['Correctness'] = correctness[i]
            row_['DataCoverage'] = coverage[i]
            row_['Fluency'] = fluency[i]
            row_['Relevance'] = relevance[i]
            row_['TextStructure'] = structure[i]
            data_discretized.append(row_)
            break

data_discretized = sorted(data_discretized, key=lambda x: x['id'])         

Computing the Fleiss' Kappa agreements

In [4]:
assignments = set([(w['submission_id'], w['sample_id']) for w in data_discretized])

correctness = np.zeros((len(assignments), n_cat))
coverage = np.zeros((len(assignments), n_cat))
fluency = np.zeros((len(assignments), n_cat))
relevance = np.zeros((len(assignments), n_cat))
structure = np.zeros((len(assignments), n_cat))

for i, (submission_id, sample_id) in enumerate(assignments):
    fdata = [w for w in data_discretized if w['submission_id'] == submission_id and w['sample_id'] == sample_id]
    
    for rating in fdata:
        correctness[i, rating['Correctness']-1] += 1
        coverage[i, rating['DataCoverage']-1] += 1
        fluency[i, rating['Fluency']-1] += 1
        relevance[i, rating['Relevance']-1] += 1
        structure[i, rating['TextStructure']-1] += 1      
        
pd.DataFrame({"Fleiss' Kappa": {
    'Correctness': fleiss_kappa(correctness),
    'Data Coverage': fleiss_kappa(coverage),
    'Fluency': fleiss_kappa(fluency),
    'Relevance': fleiss_kappa(relevance),
    'Text Structure': fleiss_kappa(structure),
}}).round(3)

,Fleiss' Kappa
Correctness,0.059
Data Coverage,0.068
Fluency,0.058
Relevance,0.041
Text Structure,0.043


# Human Evaluation

Results of the human evaluation for the participating systems according to original ratings of correctness, data coverage, fluency, relevance and text structure.

In [5]:
df = pd.DataFrame(data)

submissions = df.groupby("submission_id")["Correctness", "DataCoverage", "Fluency", "Relevance", "TextStructure"]
submissions.agg([np.mean, np.std]).sort_values(by=('Correctness', 'mean'), ascending=False).round(3)

/usr/local/lib/python3.7/site-packages/ipykernel_launcher.py:3: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  This is separate from the ipykernel package so we can avoid doing imports until


Correctness         DataCoverage         Fluency          \
                      mean     std         mean     std    mean     std   
submission_id                                                             
NUIG-DSI            87.228  17.713       87.582  17.672  84.275  19.176   
RALI                87.227  17.804       89.854  16.756  75.700  24.612   
REF                 87.082  18.492       88.983  17.266  83.436  21.120   
FBConvAI            86.725  18.904       87.459  19.183  85.067  19.971   
AmazonAI            86.448  18.478       88.064  17.392  84.116  18.601   
bt5                 86.335  19.339       86.689  20.535  82.303  20.963   
OSU_Neural_NLG      86.060  18.243       88.345  17.069  83.554  20.335   
cuni                85.824  19.973       88.352  17.698  82.951  20.854   
DANGNT-SGU          85.105  19.741       88.807  17.253  73.496  25.037   
CycleGT             84.949  21.072       85.865  21.241  81.116  22.530   
BASELINE2017        84.483  18.537       86.483  16.958  77.603  21.326   
BASELINE            84.120  20.425       86.079  19.396  77.429  23.309   
TGen                83.994  21.569       83.747  22.637  81.678  21.699   
Huawei              76.974  24.564       80.564  23.363  70.993  26.269   
ORANGE-NLG          73.341  27.364       77.099  25.532  74.002  25.590   
NILC                73.114  28.788       78.176  26.223  72.517  28.032   
UPC-POE             73.079  26.008       75.180  24.412  71.185  27.469   

               Relevance         TextStructure          
                    mean     std          mean     std  
submission_id                                           
NUIG-DSI          88.536  17.226        86.676  17.769  
RALI              89.543  16.858        79.298  22.498  
REF               87.687  17.527        85.816  19.220  
FBConvAI          88.036  18.515        87.165  17.857  
AmazonAI          88.449  16.870        86.067  18.081  
bt5               88.184  17.774        85.637  18.681  
OSU_Neural_NLG    87.019  18.192        86.307  17.361  
cuni              89.090  16.863        85.227  19.559  
DANGNT-SGU        87.436  17.687        77.878  22.569  
CycleGT           88.155  19.196        84.732  19.921  
BASELINE2017      88.178  15.267        81.633  18.895  
BASELINE          85.912  19.687        81.258  21.417  
TGen              87.097  19.581        83.899  20.770  
Huawei            80.938  23.792        75.092  24.578  
ORANGE-NLG        75.858  26.335        77.721  23.872  
NILC              79.116  26.197        76.972  25.465  
UPC-POE           79.185  24.331        76.185  25.958

# Human Evaluation (Z-Scores)

Results of the human evaluation for the participating systems according to normalized z-scores for correctness, data coverage, fluency, relevance and text structure.

In [6]:
data_ = []
worker_ids = set([w['worker_id'] for w in data])
for worker_id in worker_ids:
    fdata = [w for w in data if w['worker_id'] == worker_id]
    
    ids = [w['id'] for w in fdata]
    correctness = zscore([w['Correctness'] for w in fdata])
    coverage = zscore([w['DataCoverage'] for w in fdata])
    fluency = zscore([w['Fluency'] for w in fdata])
    relevance = zscore([w['Relevance'] for w in fdata])
    structure = zscore([w['TextStructure'] for w in fdata])
    
    for i, id_ in enumerate(ids):
        for j, row in enumerate(data):
            if row['id'] == id_:
                row_ = copy.copy(row)
                row_['Correctness'] = correctness[i]
                row_['DataCoverage'] = coverage[i]
                row_['Fluency'] = fluency[i]
                row_['Relevance'] = relevance[i]
                row_['TextStructure'] = structure[i]
                data_.append(row_)
                break
                
df = pd.DataFrame(data_)

submissions = df.groupby("submission_id")["Correctness", "DataCoverage", "Fluency", "Relevance", "TextStructure"]
submissions.agg([np.mean, np.std]).sort_values(by=('Correctness', 'mean'), ascending=False).round(3)

/usr/local/lib/python3.7/site-packages/scipy/stats/stats.py:2315: RuntimeWarning: invalid value encountered in true_divide
  return (a - mns) / sstd
/usr/local/lib/python3.7/site-packages/ipykernel_launcher.py:27: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.


Correctness        DataCoverage        Fluency         \
                      mean    std         mean    std    mean    std   
submission_id                                                          
AmazonAI             0.217  0.725        0.207  0.696   0.310  0.723   
REF                  0.181  0.762        0.173  0.705   0.181  0.828   
OSU_Neural_NLG       0.180  0.726        0.216  0.698   0.229  0.796   
NUIG-DSI             0.180  0.770        0.082  0.978   0.200  0.761   
RALI                 0.179  0.799        0.269  0.679  -0.156  1.048   
bt5                  0.146  0.770        0.060  0.888   0.130  0.839   
DANGNT-SGU           0.136  0.754        0.228  0.629  -0.162  1.072   
BASELINE             0.131  0.793        0.143  0.772   0.022  0.913   
FBConvAI             0.131  0.857        0.077  0.939   0.243  0.769   
cuni                 0.114  0.862        0.149  0.760   0.154  0.861   
BASELINE2017         0.079  0.779        0.096  0.817  -0.082  1.025   
TGen                 0.068  0.914       -0.080  1.077   0.147  0.892   
CycleGT              0.025  0.950       -0.027  0.978   0.039  0.951   
Huawei              -0.238  1.209       -0.182  1.176  -0.306  1.199   
ORANGE-NLG          -0.484  1.470       -0.454  1.378  -0.228  1.113   
NILC                -0.509  1.423       -0.362  1.392  -0.318  1.284   
UPC-POE             -0.536  1.342       -0.597  1.362  -0.402  1.269   

               Relevance        TextStructure         
                    mean    std          mean    std  
submission_id                                         
AmazonAI           0.194  0.678         0.240  0.738  
REF                0.112  0.717         0.162  0.806  
OSU_Neural_NLG     0.110  0.814         0.238  0.740  
NUIG-DSI           0.114  0.790         0.221  0.749  
RALI               0.154  0.780        -0.211  1.152  
bt5                0.134  0.658         0.177  0.739  
DANGNT-SGU         0.154  0.592        -0.153  1.087  
BASELINE           0.097  0.809         0.047  0.902  
FBConvAI           0.056  0.851         0.234  0.693  
cuni               0.147  0.755         0.146  0.804  
BASELINE2017       0.194  0.792         0.028  1.003  
TGen               0.095  0.752         0.095  0.859  
CycleGT            0.095  0.857         0.043  0.910  
Huawei            -0.247  1.369        -0.302  1.218  
ORANGE-NLG        -0.559  1.625        -0.258  1.180  
NILC              -0.406  1.440        -0.336  1.324  
UPC-POE           -0.442  1.392        -0.371  1.322

# Statistical Testing

## Wilcoxon rank-sum significant test

In [17]:
def parse(data):
    data = sorted(data, key=lambda x: (x['submission_id'], int(x['sample_id'])))

    correctness, coverage, fluency, relevance, structure = {}, {}, {}, {}, {}
    for row in data:
        submission_id = row['submission_id']
        if submission_id not in correctness:
            correctness[submission_id] = []
            coverage[submission_id] = []
            fluency[submission_id] = []
            relevance[submission_id] = []
            structure[submission_id] = []

        correctness[submission_id].append(row['Correctness'])
        coverage[submission_id].append(row['DataCoverage'])
        fluency[submission_id].append(row['Fluency'])
        relevance[submission_id].append(row['Relevance'])
        structure[submission_id].append(row['TextStructure'])

    # convert NaN values to zero
    for submission_id in correctness:
        correctness[submission_id] = np.nan_to_num(correctness[submission_id])
        coverage[submission_id] = np.nan_to_num(coverage[submission_id])
        fluency[submission_id] = np.nan_to_num(fluency[submission_id])
        relevance[submission_id] = np.nan_to_num(relevance[submission_id])
        structure[submission_id] = np.nan_to_num(structure[submission_id])
    return correctness, coverage, fluency, relevance, structure
    
def rank_systems(X, name):
    submissions = sorted(X.keys(), key=lambda x: np.mean(X[x]), reverse=True)
    ranking = { s:1 for i, s in enumerate(submissions) }

    for rank, subA in enumerate(submissions):
        for subB in submissions[rank+1:]:
            s, pvalue = ranksums(X[subA], X[subB])
            if pvalue < 0.05:
                ranking[subB] = ranking[subA] + 1
            else:
                ranking[subB] = ranking[subA]  

    ranking_ = {}
    for sub in ranking:
        rank = ranking[sub]
        mean = np.mean(X[sub])
        ranking_[sub] = { 'Ranking': int(rank), name: round(mean, 3) }

    return ranking_

correctness, coverage, fluency, relevance, structure = parse(data_)


### All Data

In [18]:
pd.DataFrame(rank_systems(correctness, 'Correctness')).T

,Ranking,Correctness
AmazonAI,1.0,0.216
REF,1.0,0.181
OSU_Neural_NLG,1.0,0.180
NUIG-DSI,1.0,0.180
RALI,1.0,0.179
bt5,1.0,0.146
DANGNT-SGU,1.0,0.136
BASELINE,1.0,0.131
FBConvAI,1.0,0.131
cuni,1.0,0.114


In [19]:
pd.DataFrame(rank_systems(coverage, 'Coverage')).T

,Ranking,Coverage
RALI,1.0,0.268
DANGNT-SGU,1.0,0.228
OSU_Neural_NLG,1.0,0.215
AmazonAI,1.0,0.207
REF,1.0,0.173
cuni,1.0,0.149
BASELINE,1.0,0.143
BASELINE2017,1.0,0.096
NUIG-DSI,1.0,0.082
FBConvAI,1.0,0.077


In [20]:
pd.DataFrame(rank_systems(fluency, 'Fluency')).T

,Ranking,Fluency
AmazonAI,1.0,0.310
FBConvAI,1.0,0.243
OSU_Neural_NLG,1.0,0.229
NUIG-DSI,1.0,0.200
REF,1.0,0.181
cuni,1.0,0.154
TGen,1.0,0.147
bt5,1.0,0.130
CycleGT,1.0,0.039
BASELINE,1.0,0.022


In [21]:
pd.DataFrame(rank_systems(relevance, 'Relevance')).T


,Ranking,Relevance
AmazonAI,1.0,0.193
BASELINE2017,1.0,0.193
DANGNT-SGU,1.0,0.153
RALI,1.0,0.153
cuni,1.0,0.146
bt5,1.0,0.134
NUIG-DSI,1.0,0.113
REF,1.0,0.112
OSU_Neural_NLG,1.0,0.110
BASELINE,1.0,0.097


In [22]:
pd.DataFrame(rank_systems(structure, 'Text Structure')).T
    

,Ranking,Text Structure
AmazonAI,1.0,0.240
OSU_Neural_NLG,1.0,0.238
FBConvAI,1.0,0.234
NUIG-DSI,1.0,0.221
bt5,1.0,0.177
REF,1.0,0.162
cuni,1.0,0.146
TGen,1.0,0.095
BASELINE,1.0,0.047
CycleGT,1.0,0.043


### Domain Type 1

In [23]:
fdata = [w for w in data_ if w['domain'] == 'type1']
correctness, coverage, fluency, relevance, structure = parse(fdata)


In [24]:
pd.DataFrame(rank_systems(correctness, 'Correctness')).T


,Ranking,Correctness
AmazonAI,1.0,0.273
FBConvAI,1.0,0.256
bt5,1.0,0.250
ORANGE-NLG,1.0,0.241
RALI,1.0,0.221
REF,1.0,0.194
cuni,1.0,0.181
NUIG-DSI,1.0,0.164
OSU_Neural_NLG,1.0,0.153
BASELINE,1.0,0.133


In [25]:
pd.DataFrame(rank_systems(coverage, 'Coverage')).T


,Ranking,Coverage
AmazonAI,1.0,0.269
cuni,1.0,0.248
RALI,1.0,0.247
DANGNT-SGU,1.0,0.203
REF,1.0,0.201
BASELINE,1.0,0.166
NILC,1.0,0.149
bt5,1.0,0.144
OSU_Neural_NLG,1.0,0.133
ORANGE-NLG,1.0,0.115


In [26]:
pd.DataFrame(rank_systems(fluency, 'Fluency')).T


,Ranking,Fluency
FBConvAI,1.0,0.336
AmazonAI,1.0,0.258
NUIG-DSI,1.0,0.240
cuni,1.0,0.202
bt5,1.0,0.190
ORANGE-NLG,1.0,0.172
OSU_Neural_NLG,1.0,0.157
REF,1.0,0.146
NILC,1.0,0.106
Huawei,1.0,0.087


In [27]:
pd.DataFrame(rank_systems(relevance, 'Relevance')).T


,Ranking,Relevance
cuni,1.0,0.222
bt5,1.0,0.198
NUIG-DSI,1.0,0.181
AmazonAI,1.0,0.174
REF,1.0,0.149
NILC,1.0,0.144
ORANGE-NLG,1.0,0.141
RALI,1.0,0.121
TGen,1.0,0.093
DANGNT-SGU,1.0,0.093


In [28]:
pd.DataFrame(rank_systems(structure, 'Text Structure')).T
    

,Ranking,Text Structure
bt5,1.0,0.271
FBConvAI,1.0,0.245
NUIG-DSI,1.0,0.234
cuni,1.0,0.219
AmazonAI,1.0,0.218
OSU_Neural_NLG,1.0,0.213
ORANGE-NLG,1.0,0.115
NILC,1.0,0.105
REF,1.0,0.100
UPC-POE,1.0,0.082


### Domain Type 2

In [29]:
fdata = [w for w in data_ if w['domain'] == 'type2']
correctness, coverage, fluency, relevance, structure = parse(fdata)


In [30]:
pd.DataFrame(rank_systems(correctness, 'Correctness')).T


,Ranking,Correctness
OSU_Neural_NLG,1.0,0.220
BASELINE2017,1.0,0.214
DANGNT-SGU,1.0,0.210
RALI,1.0,0.200
AmazonAI,1.0,0.193
BASELINE,1.0,0.180
NUIG-DSI,1.0,0.171
cuni,1.0,0.160
CycleGT,1.0,0.132
TGen,1.0,0.125


In [31]:
pd.DataFrame(rank_systems(coverage, 'Coverage')).T


,Ranking,Coverage
BASELINE,1.0,0.271
RALI,1.0,0.266
DANGNT-SGU,1.0,0.220
REF,1.0,0.217
OSU_Neural_NLG,1.0,0.214
AmazonAI,1.0,0.205
cuni,1.0,0.190
BASELINE2017,1.0,0.136
NUIG-DSI,1.0,0.101
TGen,1.0,0.089


In [32]:
pd.DataFrame(rank_systems(fluency, 'Fluency')).T


,Ranking,Fluency
AmazonAI,1.0,0.397
OSU_Neural_NLG,1.0,0.301
CycleGT,1.0,0.242
bt5,1.0,0.218
TGen,1.0,0.218
FBConvAI,1.0,0.216
NUIG-DSI,1.0,0.202
REF,1.0,0.151
cuni,1.0,0.138
BASELINE,1.0,0.072


In [33]:
pd.DataFrame(rank_systems(relevance, 'Relevance')).T


,Ranking,Relevance
BASELINE2017,1.0,0.244
DANGNT-SGU,1.0,0.231
AmazonAI,1.0,0.208
cuni,1.0,0.194
OSU_Neural_NLG,1.0,0.189
BASELINE,1.0,0.185
CycleGT,1.0,0.174
REF,1.0,0.174
NUIG-DSI,1.0,0.126
RALI,1.0,0.124


In [34]:
pd.DataFrame(rank_systems(structure, 'Text Structure')).T


,Ranking,Text Structure
NUIG-DSI,1.0,0.324
OSU_Neural_NLG,1.0,0.279
CycleGT,1.0,0.250
FBConvAI,1.0,0.211
AmazonAI,1.0,0.208
bt5,1.0,0.199
cuni,1.0,0.193
REF,1.0,0.166
TGen,1.0,0.163
BASELINE,1.0,0.081


### Domain Type 3

In [35]:
fdata = [w for w in data_ if w['domain'] == 'type3']
correctness, coverage, fluency, relevance, structure = parse(fdata)


In [36]:
pd.DataFrame(rank_systems(correctness, 'Correctness')).T


,Ranking,Correctness
REF,1.0,0.208
NUIG-DSI,1.0,0.193
AmazonAI,1.0,0.191
OSU_Neural_NLG,1.0,0.181
DANGNT-SGU,1.0,0.155
RALI,1.0,0.144
bt5,1.0,0.129
BASELINE,1.0,0.109
FBConvAI,1.0,0.090
BASELINE2017,1.0,0.067


In [37]:
pd.DataFrame(rank_systems(coverage, 'Coverage')).T


,Ranking,Coverage
RALI,1.0,0.282
OSU_Neural_NLG,1.0,0.267
DANGNT-SGU,1.0,0.248
AmazonAI,1.0,0.169
REF,1.0,0.137
BASELINE2017,1.0,0.114
BASELINE,1.0,0.075
CycleGT,1.0,0.073
cuni,1.0,0.070
NUIG-DSI,1.0,0.058


In [38]:
pd.DataFrame(rank_systems(fluency, 'Fluency')).T


,Ranking,Fluency
AmazonAI,1.0,0.305
OSU_Neural_NLG,1.0,0.242
REF,1.0,0.215
FBConvAI,1.0,0.196
NUIG-DSI,1.0,0.174
TGen,1.0,0.155
cuni,1.0,0.130
bt5,1.0,0.055
CycleGT,1.0,0.034
BASELINE,1.0,-0.022


In [39]:
pd.DataFrame(rank_systems(relevance, 'Relevance')).T


,Ranking,Relevance
BASELINE2017,1.0,0.253
AmazonAI,1.0,0.198
RALI,1.0,0.185
DANGNT-SGU,1.0,0.158
bt5,1.0,0.123
BASELINE,1.0,0.099
OSU_Neural_NLG,1.0,0.091
TGen,1.0,0.086
CycleGT,1.0,0.086
cuni,1.0,0.079


In [40]:
pd.DataFrame(rank_systems(structure, 'Text Structure')).T


,Ranking,Text Structure
AmazonAI,1.0,0.267
FBConvAI,1.0,0.236
OSU_Neural_NLG,1.0,0.235
REF,1.0,0.199
NUIG-DSI,1.0,0.169
bt5,1.0,0.109
TGen,1.0,0.090
cuni,1.0,0.080
CycleGT,1.0,0.062
BASELINE2017,1.0,0.055
